# Mice Data and setup:

In [ ]:
#!/bin/bash
set -e

# Clone and setup (skip if already done)
if [ ! -d "BERNN_MSMS" ]; then
  git clone https://github.com/spell00/BERNN_MSMS.git
fi
cd BERNN_MSMS

# Create virtual environment if not exists
if [ ! -d "bernn_env" ]; then
  python3.10 -m venv bernn_env
fi
source bernn_env/bin/activate

# Install dependencies
pip install --upgrade pip setuptools wheel
pip install -r requirements.txt
pip install -e .

# Run training on the built-in AgingMice dataset
python bernn/dl/train/train_ae_classifier_holdout.py \
  --device=cuda:0 \
  --dataset=mice \
  --n_trials=1 \
  --n_repeats=1 \
  --n_epochs=10 \
  --early_stop=5 \
  --exp_id=test_mice \
  --path=data \
  --groupkfold=1 \
  --dloss=DANN \
  --variational=0 \
  --kan=0 \
  --tied_weights=0 \
  --rec_loss=l1 \
  --pool=0 \
  --log_mlflow=0 \
  --log_neptune=0 \
  --log_tb=0 \
  --log_metrics=0 \
  --keep_models=0

# Alzheimer with train_ae_classifier_holdout:

In [ ]:
#!/bin/bash
set -e

# Run training on the built-in Alzheimer dataset --- dloss: inverseTriplet DANN revTriplet normae no 
python bernn/dl/train/train_ae_classifier_holdout.py \
  --device=cuda:0 \
  --dataset=alzheimer \
  --n_trials=1 \
  --n_repeats=1 \
  --n_epochs=10 \
  --early_stop=5 \
  --early_warmup_stop=10 \
  --exp_id=test_alzheimer \
  --path=data/Alzheimer \
  --csv_file=unique_genes.csv \
  --groupkfold=1 \
  --embeddings_meta=2 \
  --n_meta=2 \
  --dloss=DANN \
  --variational=0 \
  --kan=0 \
  --tied_weights=0 \
  --rec_loss=l1 \
  --pool=1 \
  --use_l1=0 \
  --prune_network=0 \
  --log_mlflow=0 \
  --log_neptune=0 \
  --log_tb=0 \
  --log_metrics=0 \
  --log_plots=0 \
  --keep_models=0

# Alzheimer with train_ae_then_classifier_holdout:

# Use Docker

In [ ]:
# Run training inside the container
# - Mounts the current repo directory into /app
# - Uses CPU (remove --device=cpu and add --gpus all for GPU)
docker run --rm \
  -v "$(pwd)":/app \
  -w /app \
  bernn_msms:latest \
  python bernn/dl/train/train_ae_classifier_holdout.py .....

In [2]:
import pandas as pd
subjects = pd.read_csv("BERNN_MSMS/data/Alzheimer_Test/subjects_experiment_ATN_verified_diagnosis.csv")
# contaminants = pd.read_csv("BERNN_MSMS/data/Alzheimer_Test/contaminants.csv")
unique_genes = pd.read_csv("BERNN_MSMS/data/Alzheimer_Test/unique_genes.csv")

In [3]:
print(subjects.shape)
print(subjects['Batch'].nunique())
subjects

(979, 6)
22


,sample_id,Batch,short_id,Gender,Age at time of LP (yrs),ATN_diagnosis
0,Batch-00_1287-1_1r,0,1287-1_1r,Male,81.0,NPH
1,Batch-00_1287-1_2,0,1287-1_2,Male,81.0,NPH
2,Batch-03_1187-1_1,3,1187-1_1,Female,82.0,NPH
3,Batch-03_1187-1_2,3,1187-1_2,Female,82.0,NPH
4,Batch-03_1187-2_1,3,1187-2_1,Female,82.0,NPH
...,...,...,...,...,...,...
974,Batch-23_4617-3_2,23,4617-3_2,Male,41.0,NaN
975,Batch-23_Pool-45_1,23,Pool-45_1,NaN,NaN,NaN
976,Batch-23_Pool-45_2,23,Pool-45_2,NaN,NaN,NaN
977,Batch-23_Pool-46_1,23,Pool-46_1,NaN,NaN,NaN


In [4]:
print(unique_genes.T.shape)
unique_genes.T.iloc[:,:10]

(980, 896)


,0,1,2,3,4,5,6,7,8,9
Gene_id,Gene_1,Gene_2,Gene_3,Gene_4,Gene_5,Gene_6,Gene_7,Gene_8,Gene_9,Gene_10
Batch-00_1287-1_1r,12413900.0,9109750.0,673018.0,NaN,NaN,NaN,NaN,NaN,NaN,77590.9
Batch-00_1287-1_2,12789900.0,9507820.0,524128.0,NaN,NaN,NaN,NaN,NaN,NaN,78563.9
Batch-03_1187-1_1,6822240.0,12788800.0,515014.0,NaN,NaN,NaN,100881.0,103310.0,NaN,103096.0
Batch-03_1187-1_2,7040870.0,12436400.0,529359.0,NaN,NaN,NaN,91186.8,146627.0,NaN,81333.5
...,...,...,...,...,...,...,...,...,...,...
Batch-23_4617-3_2,6166740.0,8706150.0,382704.0,NaN,NaN,NaN,NaN,NaN,NaN,84788.3
Batch-23_Pool-45_1,10532700.0,12090800.0,323157.0,NaN,NaN,NaN,NaN,NaN,NaN,73999.2
Batch-23_Pool-45_2,10565100.0,11905200.0,288226.0,NaN,NaN,NaN,NaN,NaN,NaN,90129.9
Batch-23_Pool-46_1,9953950.0,11476200.0,334149.0,NaN,NaN,NaN,59534.8,NaN,NaN,73079.9


In [6]:
import torch 
model_dir = "/mnt/scratch1/ashkanm/move2scratch/mithlesh/BERNN/BERNN_MSMS/logs/ae_then_classifier_holdout/a336f174-0a24-41dc-ab78-ca80b17a4c16/warmup.pth"
model = torch.load(model_dir)

In [7]:
model.keys()

odict_keys(['enc.layers.0.0.weight', 'enc.layers.0.0.bias', 'enc.layers.0.1.weight', 'enc.layers.0.1.bias', 'enc.layers.0.1.running_mean', 'enc.layers.0.1.running_var', 'enc.layers.0.1.num_batches_tracked', 'enc.layers.1.0.weight', 'enc.layers.1.0.bias', 'dec.layers.layer1.0.weight', 'dec.layers.layer1.0.bias', 'dec.layers.layer1.1.weight', 'dec.layers.layer1.1.bias', 'dec.layers.layer1.1.running_mean', 'dec.layers.layer1.1.running_var', 'dec.layers.layer1.1.num_batches_tracked', 'dec.layers.layer2.0.weight', 'dec.layers.layer2.0.bias', 'mapper.net.0.weight', 'mapper.net.0.bias', 'mapper.net.1.weight', 'mapper.net.1.bias', 'mapper.net.1.running_mean', 'mapper.net.1.running_var', 'mapper.net.1.num_batches_tracked', 'mapper.net.4.weight', 'mapper.net.4.bias', 'gaussian_sampling.mu.weight', 'gaussian_sampling.mu.bias', 'gaussian_sampling.log_var.weight', 'gaussian_sampling.log_var.bias', 'dann_discriminator.linear1.0.weight', 'dann_discriminator.linear1.0.bias', 'dann_discriminator.linear